In [ ]:
# ================================================================
# 1f. Wireframe close-up — one slice, finer stride, to spot local discontinuities.
#     Wireframe shows every polygon edge so sharp kinks are immediately visible.
#     Red edges = gradient > threshold → likely discontinuity.
# ================================================================

CLOSEUP_K = 10           # which slice
CLOSEUP_STRIDE = 8       # finer than the surface plots
GRAD_THRESH = 0.15       # z-change per moving-pixel — flag as "jagged"

z_close = z_ref_all[::CLOSEUP_STRIDE, ::CLOSEUP_STRIDE, CLOSEUP_K]
Xc, Yc = z_close.shape

# Compute gradient on downsampled surface
gx_close = sobel(z_close, axis=0, mode='nearest')
gy_close = sobel(z_close, axis=1, mode='nearest')
grad_close = np.sqrt(gx_close**2 + gy_close**2)

# Find edges where gradient is large
jagged_edges = grad_close > GRAD_THRESH
print(f"Slice k={CLOSEUP_K}: {jagged_edges.sum()} / {Xc*Yc} grid cells exceed gradient threshold {GRAD_THRESH} "
      f"({100*jagged_edges.sum()/(Xc*Yc):.1f}%)")

# Plot: wireframe with color-coded edges
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

x_c = np.arange(0, X, CLOSEUP_STRIDE, dtype=np.float32)
y_c = np.arange(0, Y, CLOSEUP_STRIDE, dtype=np.float32)
Xcm, Ycm = np.meshgrid(x_c, y_c, indexing='ij')

# Grey wireframe for smooth regions
z_smooth = z_close.copy()
z_smooth[jagged_edges] = np.nan
ax.plot_wireframe(Xcm, Ycm, z_smooth, color='#888888', linewidth=0.3, alpha=0.5)

# Red wireframe for jagged regions
z_jagged = z_close.copy()
z_jagged[~jagged_edges] = np.nan
ax.plot_wireframe(Xcm, Ycm, z_jagged, color='#FF3333', linewidth=0.8, alpha=0.9)

ax.set_xlabel('Moving X', fontsize=11)
ax.set_ylabel('Moving Y', fontsize=11)
ax.set_zlabel('Ref Z', fontsize=11)
ax.set_title(f'Frame {ANALYSIS_FRAME}  k={CLOSEUP_K}  Wireframe Close-up  '
             f'(red = grad > {GRAD_THRESH} px/px)  stride={CLOSEUP_STRIDE}',
             fontsize=13, fontweight='bold')
ax.view_init(elev=30, azim=-45)
fig.tight_layout()
plt.show()

print("If red edges form isolated spikes → surface has local discontinuities → causes XY gaps.")
print("If red edges form coherent patches → that region of the surface is genuinely steep (normal).")

In [ ]:
# ================================================================
# 1e. Multi-slice overlay — all K slices on ONE 3D axis, each a different colour.
#     Lets you see the spatial relationship between moving slices at a glance.
# ================================================================

STRIDE_OVERLAY = 30  # heavier downsample for overlay

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

x_mov = np.arange(0, X, STRIDE_OVERLAY, dtype=np.float32)
y_mov = np.arange(0, Y, STRIDE_OVERLAY, dtype=np.float32)
Xm, Ym = np.meshgrid(x_mov, y_mov, indexing='ij')

colors = cm.tab20(np.linspace(0, 1, K_local))

for k_idx in range(K_local):
    z_surf = z_ref_all[::STRIDE_OVERLAY, ::STRIDE_OVERLAY, k_idx]
    xref_ds = phase_new[::STRIDE_OVERLAY, ::STRIDE_OVERLAY, k_idx, 0]
    yref_ds = phase_new[::STRIDE_OVERLAY, ::STRIDE_OVERLAY, k_idx, 1]
    valid_ds = (
        (xref_ds >= 0) & (xref_ds < F260517_ref_mem.shape[2]) &
        (yref_ds >= 0) & (yref_ds < F260517_ref_mem.shape[1]) &
        np.isfinite(z_surf)
    )
    z_display = z_surf.copy()
    z_display[~valid_ds] = np.nan
    
    ax.plot_surface(Xm, Ym, z_display,
                    color=colors[k_idx], edgecolor='none',
                    alpha=0.35, antialiased=True)

ax.set_xlabel('Moving X', fontsize=11)
ax.set_ylabel('Moving Y', fontsize=11)
ax.set_zlabel('Ref Z', fontsize=11)
ax.set_title(f'Frame {ANALYSIS_FRAME} — All {K_local} slices overlaid  (stride={STRIDE_OVERLAY})',
             fontsize=14, fontweight='bold')
ax.view_init(elev=20, azim=-50)
fig.tight_layout()
plt.show()

print("Each colour = one moving slice k. Surfaces should be roughly parallel sheets.")
print("If they cross or have sharp local deviations → surface is not smooth locally.")

# Projection Black-Hole Diagnosis

Runs registration for one ref-update block (frames 20–39), then analyses **why**
projection holes (fill_value pixels) appear on the fixed target planes.

Two diagnostic outputs:
1. **Surface visualisation** — z-coordinate of phase_new per moving slice k, plus
   XY gradient magnitude to check smoothness.
2. **Hole classification** — per target plane, split uncovered pixels into:
   - **XY gap**: no moving pixel maps within 1 px in XY (surface doesn't cover this region)
   - **Z gap**: moving pixels DO map nearby in XY, but their z is outside z_window

If Z gaps dominate, widen z_window.  If XY gaps dominate, the surface is not
dense enough — increase supersurface upsampling or add hole-filling.

In [ ]:
import os, sys, time
from pathlib import Path

import cupy as cp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.ndimage import sobel, distance_transform_edt

cp.cuda.Device(1).use()

# Paths
NOTEBOOK_ROOT = Path("/home/cyf/wbi/Virginia/code/wbi_0123/wholistic_registration/src/wholistic_registration")
sys.path.insert(0, str(NOTEBOOK_ROOT))
COARSEFLOW_ROOT = Path("/home/cyf/wbi/Virginia/code/CoarseFlow").resolve()
sys.path.insert(0, str(COARSEFLOW_ROOT))

from utils import IO, calFlowCrossResolution, mask, preprocess as prep
from utils.calFlowCrossResolution import project_coords_to_fixed_planes_gpu

# Import helpers from tests/ dir
HELPERS_DIR = NOTEBOOK_ROOT / "tests"
sys.path.insert(0, str(HELPERS_DIR))
import f260517_helpers as fh

print("Imports OK.")

In [ ]:
# ================================================================
# Data loading  (same as notebook)
# ================================================================
F260517_mov_path = "/home/cyf/wbi/Virginia/raw_data/f260517/260517_exp_00001_TZCYX.ome.tiff"
F260517_ref_path = "/home/cyf/wbi/Virginia/raw_data/f260517/260517_anat_00003_TZCYX.ome.tiff"

F260517_mov, _ = IO.readTiff(F260517_mov_path)
F260517_ref, _ = IO.readTiff(F260517_ref_path)

F260517_ref_mem        = F260517_ref[90:310, 1, :, :]      # (Zref, Y, X)
F260517_ref_sparseCell = F260517_ref[90:310, 0, :, :]      # (Zref, Y, X)
F260517_mov_mem        = F260517_mov[:, :, 1, :, :].astype(np.float32)   # (T, K, Y, X)
F260517_mov_sparseCell = F260517_mov[:, :, 0, :, :].astype(np.float32)   # (T, K, Y, X)

print(f"ref_mem:  {F260517_ref_mem.shape}")
print(f"mov_mem:  {F260517_mov_mem.shape}")
print("Data loaded.")

In [ ]:
# ================================================================
# Setup — same parameters as run_F260517_0625.py
# ================================================================
option = {}
option["r"] = 5
option["layer"] = 3
option["iter"] = 10
option["movRange"] = 5.0
option["tol"] = 1e-6
option["zRatio_HR"] = 1
option["wrong_region_enable"] = False

thresFactor = 5.0
maskRange = [5.0, 4000.0]
smoothPenalty_raw = 0.01
percentiles = [0.1, 0.5, 1, 2, 5, 10, 25, 50, 75, 90, 95, 99, 99.5, 99.8]

# Projection params
Z_WINDOW = 3.0
SURFACE_UPSAMPLE = 2
FILL_VALUE = -200.0

# ---- z_init ----
z_init = calFlowCrossResolution.FindInitZ_stack_global_fixed_spacing(
    F260517_mov_mem[0].transpose(2, 1, 0),
    F260517_ref_mem.transpose(2, 1, 0),
    delta_ref_idx=10, use_gradient=False,
)
z_init = np.asarray(z_init, dtype=np.float32)
z_idx = np.rint(z_init).astype(np.int32)
z_idx = np.clip(z_idx, 0, F260517_ref_mem.shape[0] - 1)

K = int(z_init.shape[0])
T = int(F260517_mov_mem.shape[0])

# Identity phase
x = np.arange(F260517_mov_mem[0].shape[2], dtype=np.float32)
y = np.arange(F260517_mov_mem[0].shape[1], dtype=np.float32)
k = np.arange(K, dtype=np.int32)
X_grid, Y_grid, K_grid = np.meshgrid(x, y, k, indexing="ij")
coords_xyz = np.empty((len(x), len(y), K, 3), dtype=np.float32)
coords_xyz[..., 0] = X_grid
coords_xyz[..., 1] = Y_grid
coords_xyz[..., 2] = z_init[K_grid]
option["phase"] = coords_xyz.copy()

print(f"K={K}, T={T}")

In [ ]:
# ================================================================
# Initial reference calibration  (frames 0-4 raw moving mean)
# ================================================================
init_target = np.mean(F260517_mov_mem[[0,1,2,3,4]].astype(np.float32), axis=0)
ref_mem_adj, _, _, _ = fh.update_reference_intensity_mapping_from_target_stack(
    F260517_ref_mem=F260517_ref_mem,
    target_stack_zyx=init_target, z_idx=z_idx, option=option,
    thresFactor=thresFactor, maskRange=maskRange,
    smoothPenalty_raw=smoothPenalty_raw, percentiles=percentiles,
)
print("Initial ref calibration done.")

In [ ]:
# ================================================================
# Register frames 0–39 so we have a warm start for frame 20+
# (frame 20 inherits motion from frame 19)
# ================================================================
print("Registering frames 0–39 for warm start ...")
registered = {}  # frame -> dict

for i in range(0, 40):
    if i == 0:
        option["phase"] = coords_xyz.copy()
        option.pop("motion", None)
    
    raw_mem_zyx = F260517_mov_mem[i]  # (K,Y,X)
    mov_mem_i = raw_mem_zyx.transpose(2, 1, 0).astype(np.float32, copy=False)  # (X,Y,K)
    
    option["mask_mov"] = mask.getMask(mov_mem_i, thresFactor)
    option["mask_mov"] = mask.bwareafilt3_wei(option["mask_mov"], maskRange)
    
    phase_new, motion_current, mem_mapped = calFlowCrossResolution.getMotion_v2(
        mov_mem_i, ref_mem_adj, option, verbose=False)
    
    if hasattr(phase_new, "get"):      phase_new = phase_new.get()
    if hasattr(motion_current, "get"): motion_current = motion_current.get()
    if hasattr(mem_mapped, "get"):     mem_mapped = mem_mapped.get()
    
    phase_new      = np.asarray(phase_new, dtype=np.float32)
    motion_current = np.asarray(motion_current, dtype=np.float32)
    
    registered[i] = {
        "phase_new": phase_new,
        "motion": motion_current,
        "mem_mapped_xyk": np.asarray(mem_mapped, dtype=np.float32),
    }
    
    # Temporal initialisation
    option["motion"] = (0.7 * motion_current).astype(np.float32, copy=False)
    
    if i % 10 == 0:
        print(f"  frame {i} done")

print("Warmup registration complete.")

In [ ]:
# ================================================================
# Determine fixed_target_z from warmup frames 0-4
# ================================================================
target_z_list = []
for i in [0, 1, 2, 3, 4]:
    tz, _ = fh.estimate_projection_z_from_phase_simple(
        phase_new=registered[i]["phase_new"],
        z_init=z_init, ref_shape=F260517_ref_mem.shape,
        ref_volume_order="zyx", method="trimmed_mean",
        trim_percentiles=(5, 95), frame_idx=i,
    )
    target_z_list.append(tz)

fixed_target_z = fh.robust_average_target_z(target_z_list, method="median")
bad = ~np.isfinite(fixed_target_z)
fixed_target_z[bad] = z_init[bad]

print(f"fixed_target_z: {fixed_target_z}")
print(f"N planes: {len(fixed_target_z)}")

In [ ]:
# ================================================================
# Pick analysis frame  (e.g. frame 25 — middle of block)
# ================================================================
ANALYSIS_FRAME = 25

phase_new = registered[ANALYSIS_FRAME]["phase_new"]  # (X, Y, K, 3)  [x, y, z]
X, Y, K_local, _ = phase_new.shape

print(f"phase_new shape: {phase_new.shape}")
print(f"Moving dims: X={X}, Y={Y}, K={K_local}")

In [ ]:
# ================================================================
# Part 1: Surface Visualisation
# ================================================================

# 1a. Z-coordinate heatmaps for each moving slice k
fig, axes = plt.subplots(4, 5, figsize=(22, 14))
axes = axes.ravel()

z_ref_all = phase_new[..., 2]  # (X, Y, K) — z coordinate in reference space

for k_idx in range(K_local):
    ax = axes[k_idx]
    z_plane = z_ref_all[:, :, k_idx]  # (X, Y)
    
    # Mask invalid / out-of-bounds
    xref = phase_new[..., 0][:, :, k_idx]
    yref = phase_new[..., 1][:, :, k_idx]
    valid_xy = (
        (xref >= 0) & (xref < F260517_ref_mem.shape[2]) &
        (yref >= 0) & (yref < F260517_ref_mem.shape[1])
    )
    z_display = z_plane.copy()
    z_display[~valid_xy] = np.nan
    
    im = ax.imshow(z_display.T, origin='lower', aspect='auto',
                   cmap='viridis', interpolation='nearest')
    ax.set_title(f"k={k_idx}  z∈[{np.nanmin(z_display):.0f},{np.nanmax(z_display):.0f}]", fontsize=8)
    ax.axis('off')

fig.suptitle(f"Frame {ANALYSIS_FRAME} — Surface z-coordinate per moving slice (phase_new[...,2])",
             fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

In [ ]:
# 1b. XY gradient magnitude of the z-surface (check smoothness)
fig, axes = plt.subplots(4, 5, figsize=(22, 14))
axes = axes.ravel()

for k_idx in range(K_local):
    ax = axes[k_idx]
    z_plane = z_ref_all[:, :, k_idx].copy()
    
    # Replace nans with median for gradient computation
    nan_mask = ~np.isfinite(z_plane)
    if nan_mask.any():
        z_plane[nan_mask] = np.nanmedian(z_plane[~nan_mask])
    
    # XY gradient magnitude  (X=axis0, Y=axis1)
    gx = sobel(z_plane, axis=0, mode='nearest')
    gy = sobel(z_plane, axis=1, mode='nearest')
    grad_mag = np.sqrt(gx**2 + gy**2)  # units: ref-z pixels per moving-xy pixel
    
    im = ax.imshow(grad_mag.T, origin='lower', aspect='auto',
                   cmap='hot', interpolation='nearest',
                   vmin=0, vmax=np.percentile(grad_mag, 99))
    ax.set_title(f"k={k_idx}  max={grad_mag.max():.2f}", fontsize=8)
    ax.axis('off')

fig.suptitle(f"Frame {ANALYSIS_FRAME} — XY gradient magnitude of surface z  (large = jagged)",
             fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

# Print summary
for k_idx in range(K_local):
    z_plane = z_ref_all[:, :, k_idx]
    valid = np.isfinite(z_plane)
    if valid.any():
        gx = sobel(z_plane.copy(), axis=0, mode='nearest')
        gy = sobel(z_plane.copy(), axis=1, mode='nearest')
        g = np.sqrt(gx**2 + gy**2)
        print(f"  k={k_idx:2d}: z_range=[{np.nanmin(z_plane):.1f},{np.nanmax(z_plane):.1f}]  "
              f"grad_median={np.median(g[valid]):.3f}  grad_p99={np.percentile(g[valid], 99):.3f}")

In [ ]:
# 1c. Check: after supersurface 2x upsampling, what does the surface look like?
phase_up = fh.upsample_phase_xy_for_supersurface(phase_new, upsample_factor=SURFACE_UPSAMPLE)
Xup, Yup = phase_up.shape[0], phase_up.shape[1]
z_up = phase_up[..., 2]  # (X*2, Y*2, K)

fig, axes = plt.subplots(4, 5, figsize=(22, 14))
axes = axes.ravel()

for k_idx in range(K_local):
    ax = axes[k_idx]
    z_plane = z_up[:, :, k_idx]
    gx = sobel(z_plane, axis=0, mode='nearest')
    gy = sobel(z_plane, axis=1, mode='nearest')
    grad_mag = np.sqrt(gx**2 + gy**2)
    
    im = ax.imshow(grad_mag.T, origin='lower', aspect='auto',
                   cmap='hot', interpolation='nearest',
                   vmin=0, vmax=np.percentile(grad_mag, 99))
    ax.set_title(f"k={k_idx}  ({Xup}x{Yup})", fontsize=8)
    ax.axis('off')

fig.suptitle(f"Frame {ANALYSIS_FRAME} — Surface gradient AFTER {SURFACE_UPSAMPLE}x upsampling",
             fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 1d. 3D surface rendering — single k-slice as a spatial surface
#
# Downsample aggressively (every STRIDE pixels) for rendering performance.
# The surface is (x_mov, y_mov, z_ref) where z_ref comes from phase_new[..., 2].
# ================================================================
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm

STRIDE = 20  # downsample factor for 3D rendering (full = 1260x3000 → 63x150)

# Pick a few slices to inspect
SLICES_TO_PLOT = [0, 3, 7, 10, 13, 16, 19]  # every ~3rd slice

fig = plt.figure(figsize=(20, 16))

for sub_idx, k_idx in enumerate(SLICES_TO_PLOT):
    ax = fig.add_subplot(3, 3, sub_idx + 1, projection='3d')
    
    # Downsampled moving coordinates
    x_mov = np.arange(0, X, STRIDE, dtype=np.float32)
    y_mov = np.arange(0, Y, STRIDE, dtype=np.float32)
    Xm, Ym = np.meshgrid(x_mov, y_mov, indexing='ij')  # (Xds, Yds)
    
    # Downsampled z_ref surface
    z_surf = z_ref_all[::STRIDE, ::STRIDE, k_idx]  # (Xds, Yds)
    
    # Mask out-of-bounds points
    xref_ds = phase_new[::STRIDE, ::STRIDE, k_idx, 0]
    yref_ds = phase_new[::STRIDE, ::STRIDE, k_idx, 1]
    valid_ds = (
        (xref_ds >= 0) & (xref_ds < F260517_ref_mem.shape[2]) &
        (yref_ds >= 0) & (yref_ds < F260517_ref_mem.shape[1]) &
        np.isfinite(z_surf)
    )
    z_display = z_surf.copy()
    z_display[~valid_ds] = np.nan
    
    # Surface plot
    surf = ax.plot_surface(Xm, Ym, z_display,
                           cmap='viridis', edgecolor='none',
                           alpha=0.9, antialiased=True,
                           vmin=np.nanpercentile(z_display, 2),
                           vmax=np.nanpercentile(z_display, 98))
    
    ax.set_xlabel('Moving X', fontsize=9)
    ax.set_ylabel('Moving Y', fontsize=9)
    ax.set_zlabel('Ref Z', fontsize=9)
    ax.set_title(f'k={k_idx}  z∈[{np.nanmin(z_display):.0f},{np.nanmax(z_display):.0f}]',
                 fontsize=10)
    
    # Set a reasonable viewing angle
    ax.view_init(elev=25, azim=-60)

fig.suptitle(f'Frame {ANALYSIS_FRAME} — 3D Surface (phase_new[...,2])  downsampled {STRIDE}x',
             fontsize=14, fontweight='bold', y=1.01)
fig.tight_layout()
plt.show()

print("Note: downsampled {STRIDE}x for rendering. Full surface is {X}x{Y} pixels per slice.")
print("Smooth surface = gentle slopes. Jagged surface = sharp spikes/valleys.")

In [ ]:
# ================================================================
# Part 2: Hole Classification
#
# For each target plane m, for each hole pixel (where projection == fill_value),
# find the nearest moving voxel in XY and check:
#   - If nearest voxel's |z - target_z[m]| > z_window  → "Z gap"
#   - If nearest voxel's |z - target_z[m]| <= z_window → "XY gap"
#   - If no voxel within reasonable XY search radius       → "no coverage"
# ================================================================

# Use upsampled phase for projection
phase_use = phase_up  # (Xup, Yup, K, 3)
Xp, Yp = phase_use.shape[0], phase_use.shape[1]

# Build projection using ones (coverage map)
print("Building projection and coverage map ...")
t0 = time.time()

# Reference membrane as canvas
ref_for_proj = ref_mem_adj  # (X, Y, Zref)

# Coverage map: value=1 means at least one sample hit
coverage = calFlowCrossResolution.project_coords_to_fixed_planes_gpu(
    coords_ref_xyk_xyz=phase_use,
    ref_volume=ref_for_proj,
    target_z_planes=fixed_target_z,
    values_xyk=np.ones(phase_use.shape[:-1], dtype=np.float32),
    ref_volume_order="xyz",
    z_window=Z_WINDOW,
    downsample_xy=1,
    fill_value=0.0,
    return_numpy=True,
    output_order="zyx",
    xy_splat_mode="subpixel_footprint",
    xy_extra_radius=0,
)

if hasattr(coverage, "get"):
    coverage = coverage.get()
coverage = np.asarray(coverage, dtype=np.float32)  # (Nplanes, Yout, Xout)

Nplanes, Hy, Hx = coverage.shape
print(f"Coverage map: {coverage.shape}  (Nplanes={Nplanes}, Y={Hy}, X={Hx})")
print(f"Global uncovered ratio: {float(np.mean(coverage == 0)):.4f}")
print(f"Done in {time.time()-t0:.1f}s")

In [ ]:
# ================================================================
# Classify each hole pixel
# ================================================================

# Flatten the moving voxels' mapped coordinates - use upsampled phase
coords_flat = phase_use.reshape(-1, 3).astype(np.float32)  # (Nvox, 3) — (xref, yref, zref)
xref_flat = coords_flat[:, 0]
yref_flat = coords_flat[:, 1]
zref_flat = coords_flat[:, 2]

# Downsample the coords to output resolution (same as projection kernel)
# projection uses downsample_xy=1, so output = input resolution
xo_flat = xref_flat  # same since downsample=1
yo_flat = yref_flat

# Filter coords to those within reference bounds
Xref, Yref, Zref = ref_for_proj.shape
in_bounds = (
    (xref_flat >= 0) & (xref_flat <= Xref - 1) &
    (yref_flat >= 0) & (yref_flat <= Yref - 1) &
    (zref_flat >= 0) & (zref_flat <= Zref - 1) &
    np.isfinite(xref_flat) & np.isfinite(yref_flat) & np.isfinite(zref_flat)
)
xo_valid = xo_flat[in_bounds]
yo_valid = yo_flat[in_bounds]
zref_valid = zref_flat[in_bounds]
print(f"Moving voxels: {len(xref_flat):,} total, {in_bounds.sum():,} in-bounds")

In [ ]:
# For each target plane, classify holes.
# We use a spatial KD-tree per plane for efficient nearest-neighbour queries.
from scipy.spatial import cKDTree

XY_SEARCH_RADIUS = 2.0  # pixels — if no moving voxel within this XY radius, call it "no coverage"

hole_stats = []  # per-plane summary
hole_detail = []  # per-hole detail for selected planes

print("Classifying holes per target plane ...")
t0 = time.time()

for m in range(Nplanes):
    tz = fixed_target_z[m]
    hole_mask = coverage[m] == 0  # (Yout, Xout)
    n_holes = int(hole_mask.sum())
    
    if n_holes == 0:
        hole_stats.append({
            "plane": m, "target_z": tz, "n_holes": 0,
            "xy_gap": 0, "z_gap": 0, "no_coverage": 0,
        })
        continue
    
    # Hole pixel coordinates in output space (y, x)
    hy, hx = np.where(hole_mask)
    hole_coords = np.column_stack([hx, hy]).astype(np.float32)  # (x, y)
    
    # Build KD-tree of valid moving voxel output coordinates
    tree = cKDTree(np.column_stack([xo_valid, yo_valid]))
    
    # For each hole, find nearest moving voxel
    dists, idxs = tree.query(hole_coords, k=1, distance_upper_bound=XY_SEARCH_RADIUS)
    
    n_xy_gap = 0
    n_z_gap = 0
    n_no_cov = 0
    
    for j in range(n_holes):
        if np.isinf(dists[j]):
            # No moving voxel within XY_SEARCH_RADIUS
            n_no_cov += 1
        else:
            # Nearest voxel exists — check its z
            z_near = zref_valid[idxs[j]]
            if abs(z_near - tz) <= Z_WINDOW:
                # z is within window but pixel still uncovered → XY subpixel gap
                n_xy_gap += 1
            else:
                # z outside window
                n_z_gap += 1
    
    hole_stats.append({
        "plane": m, "target_z": tz, "n_holes": n_holes,
        "xy_gap": n_xy_gap, "z_gap": n_z_gap, "no_coverage": n_no_cov,
    })
    
    # Save per-hole detail for a few planes
    if m in [0, 5, 10, 15, 19]:
        for j in range(n_holes):
            hole_detail.append({
                "plane": m, "target_z": tz,
                "hole_x": hx[j], "hole_y": hy[j],
                "nearest_dist": float(dists[j]) if not np.isinf(dists[j]) else np.nan,
                "nearest_z": float(zref_valid[idxs[j]]) if not np.isinf(dists[j]) else np.nan,
                "category": ("xy_gap" if not np.isinf(dists[j]) and abs(zref_valid[idxs[j]] - tz) <= Z_WINDOW
                              else "z_gap" if not np.isinf(dists[j])
                              else "no_coverage"),
            })

print(f"Done in {time.time()-t0:.1f}s")

In [ ]:
# ================================================================
# Hole classification summary
# ================================================================
df_stats = pd.DataFrame(hole_stats)
df_detail = pd.DataFrame(hole_detail)

total_holes = df_stats["n_holes"].sum()
total_xy    = df_stats["xy_gap"].sum()
total_z     = df_stats["z_gap"].sum()
total_nocov = df_stats["no_coverage"].sum()

print("="*70)
print(f"Frame {ANALYSIS_FRAME} — Hole Classification Summary")
print("="*70)
print(f"Total output pixels:   {Nplanes * Hy * Hx:>12,}")
print(f"Total holes:           {total_holes:>12,}  ({100*total_holes/(Nplanes*Hy*Hx):.2f}%)")
print(f"  - XY gap:            {total_xy:>12,}  ({100*total_xy/max(total_holes,1):.1f}% of holes)")
print(f"  - Z out-of-window:   {total_z:>12,}  ({100*total_z/max(total_holes,1):.1f}% of holes)")
print(f"  - No coverage:       {total_nocov:>12,}  ({100*total_nocov/max(total_holes,1):.1f}% of holes)")
print("="*70)

if total_z > total_xy:
    print("→ VERDICT: Z gaps dominate → widen z_window or add more target planes")
elif total_xy > total_z:
    print("→ VERDICT: XY gaps dominate → surface not dense enough; increase supersurface upsampling")
else:
    print("→ VERDICT: Mixed — both XY and Z contribute similarly")

# Per-plane breakdown
print("\n--- Per-plane breakdown ---")
for _, row in df_stats.iterrows():
    n = row["n_holes"]
    if n == 0:
        print(f"  plane {row['plane']:2d}  z={row['target_z']:.0f}:  NO HOLES")
    else:
        print(f"  plane {row['plane']:2d}  z={row['target_z']:.0f}:  "
              f"holes={n:6d}  xy={row['xy_gap']/n*100:.0f}%  z={row['z_gap']/n*100:.0f}%  "
              f"nocov={row['no_coverage']/n*100:.0f}%")

In [ ]:
# ================================================================
# Visualise hole classification on a few planes
# ================================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()
sel_planes = [0, 5, 10, 15, 19]

cmap_colors = {"xy_gap": "#FF6B6B", "z_gap": "#4ECDC4", "no_coverage": "#AAAAAA"}

for idx, m in enumerate(sel_planes):
    if m >= Nplanes:
        continue
    ax = axes[idx]
    
    # Background: coverage map
    cov_plane = coverage[m]
    ax.imshow(cov_plane.T, origin='lower', aspect='auto', cmap='Greys',
              vmin=0, vmax=1, interpolation='nearest')
    
    # Overlay hole categories
    det_m = df_detail[df_detail["plane"] == m]
    for cat, color in cmap_colors.items():
        pts = det_m[det_m["category"] == cat]
        if len(pts) > 0:
            # Subsample for plotting performance
            n_show = min(len(pts), 5000)
            pts_show = pts.sample(n_show, random_state=0) if len(pts) > n_show else pts
            ax.scatter(pts_show["hole_x"], pts_show["hole_y"],
                      c=color, s=0.5, alpha=0.6, label=f"{cat} ({len(pts)})")
    
    ax.set_title(f"Plane {m}  z={fixed_target_z[m]:.0f}", fontsize=10)
    ax.set_xlim(0, Hx)
    ax.set_ylim(0, Hy)
    ax.legend(fontsize=7, markerscale=5, loc='upper right')

for idx in range(len(sel_planes), len(axes)):
    axes[idx].axis('off')

fig.suptitle(f"Frame {ANALYSIS_FRAME} — Hole classification on selected target planes",
             fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

In [ ]:
# ================================================================
# Quick check: does increasing supersurface factor reduce XY gaps?
# ================================================================
for test_factor in [1, 2, 4]:
    p_test = fh.upsample_phase_xy_for_supersurface(phase_new, upsample_factor=test_factor)
    cov_test = calFlowCrossResolution.project_coords_to_fixed_planes_gpu(
        coords_ref_xyk_xyz=p_test,
        ref_volume=ref_for_proj,
        target_z_planes=fixed_target_z,
        values_xyk=np.ones(p_test.shape[:-1], dtype=np.float32),
        ref_volume_order="xyz",
        z_window=Z_WINDOW,
        downsample_xy=1,
        fill_value=0.0,
        return_numpy=True,
        output_order="zyx",
        xy_splat_mode="subpixel_footprint",
        xy_extra_radius=0,
    )
    if hasattr(cov_test, "get"):
        cov_test = cov_test.get()
    cov_test = np.asarray(cov_test, dtype=np.float32)
    hole_frac = float(np.mean(cov_test == 0))
    print(f"  supersurface={test_factor}x  →  hole fraction = {hole_frac:.4f}  "
          f"({hole_frac*100:.2f}%)")

In [ ]:
print("\nDiagnosis complete.")
print(f"\nHole detail CSV has {len(df_detail)} rows — inspect df_detail for per-pixel data.")
print(f"Save: df_stats.to_csv('hole_stats_frame{ANALYSIS_FRAME}.csv')")